In [1]:
import pandas as pd
import numpy as np
 
 
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

La columna Size mezcla unidades: "19M" (Megabytes), "201k" (Kilobytes) y el texto "Varies with device".

## **Reto 1: La Trampa de las Unidades de Medida**
### **a. Escriban una función pura que reciba un string: si termina en 'M' lo convierte a flotante; si termina en 'k' lo convierte a flotante y lo divide entre 1024; si dice "Varies with device" devuelve np.nan.**

### **b. Apliquen la función a toda la columna con .apply().**

### **c. Ejecuten .mean(). ¿Cuál es el peso promedio en Megabytes de las apps?**

In [2]:
def convertir_size(valor):
    if valor.endswith('M'):
        return float(valor[:-1])
    if valor.endswith('k'):
        return float(valor[:-1]) / 1024
    return np.nan

df['Size'] = df['Size'].apply(convertir_size)
df['Size'].mean()

np.float64(21.51616543577433)

Reto 1: el peso promedio es 21.52 MB aprox.

La columna Last Updated guarda fechas como texto: "January 7, 2018".

## **Reto 2: El Tipo de Dato Cronológico**
### **a. Usen pd.to_datetime() para sobrescribir la columna y convertirla a datetime64.**

### **b. Creen la columna Year_Updated con df['Last Updated'].dt.year.**

### **c. Con value_counts(), ¿en qué año se actualizó la mayor cantidad de apps?**

In [3]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')
df['Year_Updated'] = df['Last Updated'].dt.year
df['Year_Updated'].value_counts()

Year_Updated
2018.0    7349
2017.0    1867
2016.0     804
2015.0     459
2014.0     209
2013.0     110
2012.0      26
2011.0      15
2010.0       1
Name: count, dtype: int64

Reto 2: 2018, con 7349 apps actualizadas.

El Reto 1 introdujo NaN en las apps cuyo tamaño decía "Varies with device".

## **Reto 3: La Decisión Arquitectónica**
### **a. Evalúen cuántos registros quedaron vacíos.**

### **b. Decidan y apliquen la mejor técnica: borrar las filas con .dropna() o imputar la mediana global del peso.**

### **c. Redacten una justificación técnica de 3 líneas.**

In [4]:
print(df['Size'].isna().sum())
df['Size'] = df['Size'].fillna(df['Size'].median())
df['Size'].describe()

1696


count    10841.000000
mean        20.183870
std         20.976262
min          0.008301
25%          5.900000
50%         13.000000
75%         26.000000
max        100.000000
Name: Size, dtype: float64

Reto 3: quedaron 1696 registros vacíos (15.6% del dataset) y elegimos imputar la mediana global (13.0 MB) en lugar de .dropna().

Justificación: borrar 15.6% de las filas desecharía datos válidos de Installs, Rating y Price, y "Varies with device" no es un error de captura sino apps modulares (grandes y populares), así que su ausencia no es aleatoria y eliminarlas metería sesgo sistemático al modelo; la mediana es robusta a los outliers de peso y conserva el tamaño de la muestra.